In [0]:
# Demo Setup: Monitoring Dashboard Usage
# This demo queries system.access.audit — a system table that logs all workspace actions.
# The setup verifies access and creates a sample dashboard to generate audit events.

import re
import json
import requests
from datetime import date, timedelta
import random
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
import time

# --- User context ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "workspace"
schema = f"demo_audit_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
# --- Check access to system.access.audit ---
# This table requires specific permissions (account admin / metastore admin / explicit GRANT).
# If learners don't have access, the instructor demonstrates live.

try:
    audit_check = spark.sql("""
        SELECT COUNT(*) AS event_count
        FROM system.access.audit
        WHERE event_date >= CURRENT_DATE() - INTERVAL 1 DAY
    """).collect()[0][0]
    print(f"\u2705 Access to system.access.audit CONFIRMED")
    print(f"   Events in last 24 hours: {audit_check}")
    audit_available = True
except Exception as e:
    error_msg = str(e)
    if "PERMISSION" in error_msg.upper() or "ACCESS" in error_msg.upper() or "DENIED" in error_msg.upper():
        print(f"\u26a0\ufe0f  No access to system.access.audit")
        print(f"   This table requires account admin or metastore admin role,")
        print(f"   or an explicit GRANT on system.access.")
        print(f"   The instructor will demonstrate this live.")
    else:
        print(f"\u26a0\ufe0f  Error checking audit table: {error_msg[:200]}")
    audit_available = False

In [0]:
# --- Create a sample Gold table and dashboard ---
# This gives us a known dashboard_id to use in the monitoring queries.

random.seed(77)

spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Northwest']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Food & Beverage']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("net_revenue", DoubleType(), False)
])

rows = []
for i in range(1, 1001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    net_rev = round(random.uniform(20, 600) * random.randint(1, 5), 2)
    rows.append((i, order_date, random.choice(regions), random.choice(categories), net_rev))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")
print(f"Created gold_sales: {df.count()} rows")

# Create and publish a small dashboard
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
parent_path = f"/Users/{username}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

table_ref = f"`{catalog}`.`{schema}`.`gold_sales`"

dashboard_def = {
    "pages": [{
        "name": "page_main",
        "displayName": "Overview",
        "layout": [{
            "widget": {
                "name": "w_title",
                "textbox_spec": "# Audit Monitoring Demo\n\nThis dashboard exists so we can monitor its usage in system.access.audit."
            },
            "position": {"x": 0, "y": 0, "width": 6, "height": 1}
        },{
            "widget": {
                "name": "w_bar",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_main", "fields": [{"name": "region", "expression": "`region`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "bar", "encodings": {"x": {"fieldName": "region", "scale": {"type": "categorical"}, "displayName": "Region"}, "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Revenue"}}}
            },
            "position": {"x": 0, "y": 1, "width": 6, "height": 4}
        }]
    }],
    "datasets": [{
        "name": "ds_main",
        "displayName": "Revenue by Region",
        "query": f"SELECT region, ROUND(SUM(net_revenue), 2) AS total_revenue FROM {table_ref} GROUP BY region ORDER BY total_revenue DESC"
    }]
}

dashboard_name = f"Audit Monitoring Demo ({clean_username}) {int(time.time())}"
resp = requests.post(
    f"https://{workspace_url}/api/2.0/lakeview/dashboards",
    headers=headers,
    json={"display_name": dashboard_name, "serialized_dashboard": json.dumps(dashboard_def), "parent_path": parent_path}
)

if resp.status_code == 200:
    dashboard_id = resp.json()["dashboard_id"]
    print(f"\n\u2705 Dashboard created: {dashboard_name}")
    print(f"   Dashboard ID: {dashboard_id}")
    # Publish it so it generates audit events
    pub_resp = requests.post(
        f"https://{workspace_url}/api/2.0/lakeview/dashboards/{dashboard_id}/published",
        headers=headers,
        json={"embed_credentials": True}
    )
    if pub_resp.status_code == 200:
        print(f"\u2705 Dashboard published")
else:
    print(f"\u26a0\ufe0f  Dashboard create returned {resp.status_code}: {resp.text[:200]}")
    dashboard_id = "<YOUR_DASHBOARD_ID>"

In [0]:
# --- Summary ---
print("SETUP COMPLETE")
print("")
print(f"User:     {username}")
print(f"Schema:   {catalog}.{schema}")
print(f"")
print(f"Table:  gold_sales - 1000 rows")
print(f"Dashboard: {dashboard_name}")
print(f"Dashboard ID: {dashboard_id}")
print(f"")
if audit_available:
    print(f"\u2705 system.access.audit is ACCESSIBLE")
    print(f"   You can run the monitoring queries directly.")
else:
    print(f"\u26a0\ufe0f  system.access.audit is NOT accessible")
    print(f"   You can still review the SQL patterns in the demo notebook.")